In [1]:
import requests
from pyspark.sql import SparkSession

def fetch_and_extract(url):
    print(f"Đang cào: {url}") # In ra để lúc test dễ nhìn
   
    try:
        response = requests.get(url, timeout=5)
       
        if response.status_code == 200:
            data = response.json()
            # Trả về từ điển chứa Data sạch
            return {
                "id": data.get("id"),
                "title": data.get("title", "Không có tên"),
                "price": data.get("price", 0.0),
                "brand": data.get("brand", "Unknown"),
                "rating": data.get("rating", 0.0)
            }
        else:
            # Trả về từ điển chứa Lỗi hệ thống
            return {"id": None, "error": f"Lỗi HTTP: {response.status_code}"}
           
    except requests.exceptions.Timeout:
        # Trả về từ điển chứa Lỗi chờ lâu
        return {"id": None, "error": "Lỗi: Quá thời gian chờ (Timeout)"}
       
    except requests.exceptions.RequestException as e:
        # Trả về từ điển chứa Lỗi sập mạng
        return {"id": None, "error": "Lỗi: Sập kết nối mạng"}


In [4]:
spark = SparkSession.builder\
            .appName("Mini_DE_Crawler_Tiki")\
            .master("local[4]")\
            .getOrCreate()

In [5]:
# 1. Chuẩn bị xấp hồ sơ thực tế (Trộn lẫn link thật và link hỏng)
danh_sach_thuc_te = [
    "https://dummyjson.com/products/1",               # Link xịn
    "https://dummyjson.com/products/99999",           # Link gây lỗi 404
    "https://dummyjson.com/products/2",               # Link xịn
    "https://web-nay-khong-the-nao-ton-tai.com",      # Link gây lỗi sập mạng
    "https://dummyjson.com/products/3"                # Link xịn
]

print("📦 Đã chuẩn bị 5 links (trong đó có 2 link chứa bom ngầm).")
print("🚨 BẮT ĐẦU CHẠY SPARK...")

# 2. Xé nhỏ giao cho công nhân (Tạo RDD)
mixed_rdd = spark.sparkContext.parallelize(danh_sach_thuc_te)

# 3. Ép công nhân đi cào và mang kết quả về
ket_qua_thuc_te = mixed_rdd.map(fetch_and_extract).collect()

# 4. Kiểm tra đống hàng hóa mang về
print("\n--- KẾT QUẢ TRẢ VỀ TỪ CÁC CÔNG NHÂN ---")
for mon_hang in ket_qua_thuc_te:
    print(mon_hang)

📦 Đã chuẩn bị 5 links (trong đó có 2 link chứa bom ngầm).
🚨 BẮT ĐẦU CHẠY SPARK...

--- KẾT QUẢ TRẢ VỀ TỪ CÁC CÔNG NHÂN ---
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'brand': 'Essence', 'rating': 2.56}
{'id': None, 'error': 'Lỗi HTTP: 404'}
{'id': 2, 'title': 'Eyeshadow Palette with Mirror', 'price': 19.99, 'brand': 'Glamour Beauty', 'rating': 2.86}
{'id': None, 'error': 'Lỗi: Sập kết nối mạng'}
{'id': 3, 'title': 'Powder Canister', 'price': 14.99, 'brand': 'Velvet Touch', 'rating': 4.64}


In [6]:
# 1. Viết bộ quy tắc cho anh bảo vệ
def la_hang_xin(mon_hang):
    """
    Hàm này kiểm tra 1 món hàng.
    Nếu trả về True -> Được đi tiếp.
    Nếu trả về False -> Bị vứt vào sọt rác.
    """
    # Nếu trong từ điển có nhãn "error", trả về False (vứt đi)
    if "error" in mon_hang:
        return False
    # Nếu không, trả về True (giữ lại)
    return True


print("🧹 Quản đốc: 'Bảo vệ đâu, ra lọc rác cho tôi!'")


# 2. Dùng .filter() để áp dụng bộ quy tắc
# mixed_rdd là cái danh sách 5 link (3 xịn, 2 hỏng) ở chặng trước
rdd_sach = mixed_rdd.map(fetch_and_extract).filter(la_hang_xin)


# 3. Kêu công nhân gom hàng sạch về (.collect())
ket_qua_sach = rdd_sach.collect()


# 4. Kiểm tra thành quả
print("\n✨ --- KHO HÀNG SAU KHI DỌN DẸP SẠCH SẼ --- ✨")
for mon_hang in ket_qua_sach:
    print(mon_hang)


print("-" * 40)
print(f"📈 Báo cáo: Đã thu được {len(ket_qua_sach)} món hàng xịn, vứt bỏ thành công 2 cục rác!")

🧹 Quản đốc: 'Bảo vệ đâu, ra lọc rác cho tôi!'

✨ --- KHO HÀNG SAU KHI DỌN DẸP SẠCH SẼ --- ✨
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'brand': 'Essence', 'rating': 2.56}
{'id': 2, 'title': 'Eyeshadow Palette with Mirror', 'price': 19.99, 'brand': 'Glamour Beauty', 'rating': 2.86}
{'id': 3, 'title': 'Powder Canister', 'price': 14.99, 'brand': 'Velvet Touch', 'rating': 4.64}
----------------------------------------
📈 Báo cáo: Đã thu được 3 món hàng xịn, vứt bỏ thành công 2 cục rác!


In [9]:
# 1. Tạo một cái "khuôn đúc" ép kiểu dữ liệu
def ep_kieu_du_lieu(mon_hang):
    """
    Hàm này nhận vào 1 món hàng sạch,
    ép nó vào đúng khuôn: id (int), price (float), title (str)...
    """
    try:
        # Nếu website trả về giá tiền dạng chữ "9.99", hàm float() sẽ biến nó thành số 9.99
        return {
            "id": int(mon_hang.get("id", 0)),
            "title": str(mon_hang.get("title", "")),
            "price": float(mon_hang.get("price", 0.0)),
            "brand": str(mon_hang.get("brand", "Unknown")),
            "rating": float(mon_hang.get("rating", 0.0))
        }
    except Exception as e:
        # Lỡ có món hàng nào bị lỗi dị thường không ép kiểu được thì báo lỗi
        return {"error": f"Lỗi ép kiểu: {str(e)}"}

print("⚙️ Quản đốc: 'Cho hàng qua máy ép khuôn để chuẩn hóa!'")

# 2. Dùng .map() để ép toàn bộ công nhân mang hàng qua máy ép kiểu
# Lưu ý: rdd_sach là kết quả từ chặng 2
rdd_chuan = rdd_sach.map(ep_kieu_du_lieu)

# 3. Lấy thử 1 món hàng đầu tiên ra kiểm tra (.take(1) thay vì .collect() hết)
hang_mau = rdd_chuan.take(1)[0]

print("\n--- KIỂM TRA CHẤT LƯỢNG SAU ÉP KHUÔN ---")
print(hang_mau)
print("-" * 30)

# Dùng hàm type() của Python để soi bằng kính lúp xem nó là chữ hay số
print(f"🔍 Kiểu dữ liệu của ID: {type(hang_mau['id'])} (Kỳ vọng: int)")
print(f"🔍 Kiểu dữ liệu của Giá: {type(hang_mau['price'])} (Kỳ vọng: float)")

⚙️ Quản đốc: 'Cho hàng qua máy ép khuôn để chuẩn hóa!'

--- KIỂM TRA CHẤT LƯỢNG SAU ÉP KHUÔN ---
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'brand': 'Essence', 'rating': 2.56}
------------------------------
🔍 Kiểu dữ liệu của ID: <class 'int'> (Kỳ vọng: int)
🔍 Kiểu dữ liệu của Giá: <class 'float'> (Kỳ vọng: float)


In [10]:
print("🚀 KÍCH HOẠT SIÊU ĐƯỜNG ỐNG (PIPELINE) SPARK...")


# Chỉ với 1 dòng code duy nhất, chúng ta gộp toàn bộ quy trình:
# Nguồn Data -> Cào -> Lọc Rác -> Ép Kiểu
pipeline_rdd = (
    spark.sparkContext.parallelize(danh_sach_thuc_te) # Nguồn link có cả xịn lẫn hỏng
    .map(fetch_and_extract)                           # Bước 1: Cào data an toàn
    .filter(la_hang_xin)                              # Bước 2: Dọn rác
    .map(ep_kieu_du_lieu)                             # Bước 3: Ép kiểu dữ liệu
)

# Kích hoạt chạy thật
thanh_pham = pipeline_rdd.collect()

print("\n🏆 --- THÀNH PHẨM CUỐI CÙNG TỪ PIPELINE ---")
for mon_hang in thanh_pham:
    print(mon_hang)


🚀 KÍCH HOẠT SIÊU ĐƯỜNG ỐNG (PIPELINE) SPARK...

🏆 --- THÀNH PHẨM CUỐI CÙNG TỪ PIPELINE ---
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'brand': 'Essence', 'rating': 2.56}
{'id': 2, 'title': 'Eyeshadow Palette with Mirror', 'price': 19.99, 'brand': 'Glamour Beauty', 'rating': 2.86}
{'id': 3, 'title': 'Powder Canister', 'price': 14.99, 'brand': 'Velvet Touch', 'rating': 4.64}


In [11]:
print("🏭 Đang đưa hàng vào máy ép Bảng tính (DataFrame)...")
# Dùng lệnh createDataFrame để biến RDD thành Spark DataFrame
df_thanh_pham = spark.createDataFrame(pipeline_rdd)
print("✅ Đã ép xong! Dữ liệu giờ đã có hình hài của một bảng Excel xịn xò.")
# Để chứng minh nó đã thành bảng, hãy in thử số cột và số dòng
so_dong = df_thanh_pham.count()
danh_sach_cot = df_thanh_pham.columns

print(f"📊 Bảng của chúng ta có {so_dong} dòng (sản phẩm) và {len(danh_sach_cot)} cột.")
print(f"🏷️ Tên các cột là: {danh_sach_cot}")

🏭 Đang đưa hàng vào máy ép Bảng tính (DataFrame)...
✅ Đã ép xong! Dữ liệu giờ đã có hình hài của một bảng Excel xịn xò.
📊 Bảng của chúng ta có 3 dòng (sản phẩm) và 5 cột.
🏷️ Tên các cột là: ['brand', 'id', 'price', 'rating', 'title']


In [12]:
print("👀 Quản đốc: 'Kéo 5 thùng hàng ra giữa sân cho tôi kiểm tra ngẫu nhiên!'")

# 1. Xem dữ liệu thực tế (In ra dạng bảng)
# truncate=False giúp chữ không bị cắt ngắn lại (thành dấu ...) nếu tên sản phẩm quá dài
df_thanh_pham.show(5, truncate=False)

print("-" * 50)
print("🔍 Quản đốc: 'Đọc bản kiểm định chất lượng (Schema) lên nghe xem!'")

# 2. Xem cấu trúc bảng (Các cột và kiểu dữ liệu)
df_thanh_pham.printSchema()


👀 Quản đốc: 'Kéo 5 thùng hàng ra giữa sân cho tôi kiểm tra ngẫu nhiên!'
+--------------+---+-----+------+-----------------------------+
|brand         |id |price|rating|title                        |
+--------------+---+-----+------+-----------------------------+
|Essence       |1  |9.99 |2.56  |Essence Mascara Lash Princess|
|Glamour Beauty|2  |19.99|2.86  |Eyeshadow Palette with Mirror|
|Velvet Touch  |3  |14.99|4.64  |Powder Canister              |
+--------------+---+-----+------+-----------------------------+

--------------------------------------------------
🔍 Quản đốc: 'Đọc bản kiểm định chất lượng (Schema) lên nghe xem!'
root
 |-- brand: string (nullable = true)
 |-- id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- title: string (nullable = true)



In [13]:
print("📦 Quản đốc: 'Anh em gom hàng lại, đóng thùng mang ra xe!'")


# 1. Đặt tên thư mục chứa file thành phẩm
ten_thu_muc_luu = "Thanh_Pham_Data"


# 2. Lệnh lưu file chuẩn Data Engineer:
# - coalesce(1): Gom lại thành 1 file duy nhất
# - write: Bắt đầu ghi
# - mode("overwrite"): Nếu chạy code lần 2, nó sẽ xóa file cũ đi ghi đè file mới lên (không bị lỗi)
# - option("header", "true"): Nhớ in kèm cái dòng tiêu đề (id, title, price...)
# - csv(): Chọn định dạng lưu là CSV


df_thanh_pham.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(ten_thu_muc_luu)


print(f"🎉 XUẤT XƯỞNG THÀNH CÔNG! Dữ liệu đã được lưu vào thư mục: {ten_thu_muc_luu}")


📦 Quản đốc: 'Anh em gom hàng lại, đóng thùng mang ra xe!'
🎉 XUẤT XƯỞNG THÀNH CÔNG! Dữ liệu đã được lưu vào thư mục: Thanh_Pham_Data


In [14]:
print("🚀 Quản đốc: 'Khởi động máy hút chân không, đóng gói chuẩn Parquet!'")
# Tên thư mục chứa file Parquet
ten_thu_muc_parquet = "Thanh_Pham_Parquet"
# Lệnh lưu Parquet cực kỳ đơn giản (Không cần option header rườm rà như CSV)
df_thanh_pham.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(ten_thu_muc_parquet)


print(f"🌟 THÀNH CÔNG RỰC RỠ! Dữ liệu đã được nén vào thư mục: {ten_thu_muc_parquet}")

🚀 Quản đốc: 'Khởi động máy hút chân không, đóng gói chuẩn Parquet!'
🌟 THÀNH CÔNG RỰC RỠ! Dữ liệu đã được nén vào thư mục: Thanh_Pham_Parquet
